usecase :
we recivied data from multiple sources (regions) we have to load data into our storage

csv with header
file name : empdata_region_datadt.csv (empdata_NAM_20260620.csv)
cid,fname,lname,age,prof
output :

cid,fullname,age,prof,load_dt,data_dt,source,created_by

mappings :

- cid->cid
- fullname -> fname+lname
- age->age
- prof ->prof
- load_dt -> current date
- data_dt -> take from the file name
- source -> take from the file name
- created by -> current user

In [0]:
import os
from pathlib import Path
from pyspark.sql.functions import col, lit, concat, current_date, current_user

vol_dir="/Volumes/we48/izdata/data/staging/BB2/uc"
for filename in os.listdir(vol_dir):
    if filename.endswith(".csv"):
        region=filename.split('_')[1]
        data_dt=filename.split('_')[2].split('.')[0]
        #print(region)
        #print(data_dt)
        df1=spark.read.format("csv").option("header",True).option("pathGlobFilter","empdata*.csv").load(vol_dir+"/"+filename)
        df2=df1.withColumn("fullname",concat(df1.fname,lit(" "),df1.lname)).withColumn("current_date",current_date()).withColumn("region",lit(region)).withColumn("data_dt",lit(data_dt)).withColumn("current_user",current_user()).drop("fname","lname")
        df3=df2.write.mode("append").format("delta").save("/Volumes/we48/izdata/data/empdata")        
        
tgt_df=spark.read.format("delta").load("/Volumes/we48/izdata/data/empdata")
tgt_df.show()